In [ ]:
import tqdm as notebook_tqdm
from datasets import Dataset
import pandas as pd
import math
from transformers import XLNetTokenizer, XLNetLMHeadModel, Trainer, TrainingArguments, pipeline

/home/shady/Desktop/project/venv/lib64/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2025-04-19 23:15:15.250031: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-04-19 23:15:15.416969: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-04-19 23:15:15.519882: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-04-19 23:15

In [3]:
df = pd.read_csv("final_dataset.csv")

# Convert to HuggingFace Dataset using only the Conversation column
dataset = Dataset.from_pandas(df[["Conversation"]].rename(columns={"Conversation": "text"}))
dataset = {"train": dataset}

In [4]:
# Step 3: Load XLNet tokenizer and model

tokenizer = XLNetTokenizer.from_pretrained("xlnet-base-cased")
model = XLNetLMHeadModel.from_pretrained("xlnet-base-cased")

/home/shady/Desktop/project/venv/lib64/python3.10/site-packages/torch/_utils.py:831: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()


In [6]:
print(dataset["train"])

Dataset({
    features: ['text'],
    num_rows: 300000
})


In [5]:
def tokenize_function(example):
    return tokenizer(example["text"])

tokenized_dataset = dataset["train"].map(tokenize_function, batched=True)

Map:   0%|          | 0/300000 [00:00<?, ? examples/s]


ValueError: Input is not valid. Should be a string, a list/tuple of strings or a list/tuple of integers.

In [6]:
# Grouping function
block_size = 128

def group_texts(examples):
    concatenated = []
    for input_ids in examples["input_ids"]:
        concatenated.extend(input_ids)

    total_length = (len(concatenated) // block_size) * block_size
    input_ids = [concatenated[i : i + block_size] for i in range(0, total_length, block_size)]
    attention_mask = [[1] * block_size for _ in input_ids]

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": input_ids.copy(),
    }

In [ ]:
# Step 4: Training in 10k chunks with evaluation
chunk_size = 10000
num_chunks = math.ceil(len(df) / chunk_size)
val_accuracies = []

for chunk_idx in range(num_chunks):
    print(f"Training chunk {chunk_idx + 1}/{num_chunks}...")

    chunk_df = df.iloc[chunk_idx * chunk_size : (chunk_idx + 1) * chunk_size]
    dataset = Dataset.from_pandas(chunk_df[["Conversation"]].rename(columns={"Conversation": "text"}))
    tokenized_dataset = dataset.map(tokenize_function, batched=True)
    lm_dataset = tokenized_dataset.map(group_texts, batched=True, remove_columns=tokenized_dataset.column_names)

    # Split into train and validation
    split_idx = int(0.9 * len(lm_dataset))
    train_dataset = lm_dataset.select(range(split_idx))
    eval_dataset = lm_dataset.select(range(split_idx, len(lm_dataset)))

    training_args = TrainingArguments(
        output_dir=f"temp/xlnet-hinglish-chunk{chunk_idx + 1}",
        evaluation_strategy="epoch",
        num_train_epochs=3,
        per_device_train_batch_size=8,
        save_steps=500,
        save_total_limit=2,
        logging_steps=100,
        warmup_steps=100,
        weight_decay=0.01,
        fp16=True,
        overwrite_output_dir=True,
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        tokenizer=tokenizer,
    )

    trainer.train()

    # Evaluate and store accuracy
    eval_result = trainer.evaluate()
    if "eval_loss" in eval_result:
        val_accuracy = 1 - eval_result["eval_loss"]
        val_accuracies.append(val_accuracy)

    # Save intermediate model
    model.save_pretrained(f"temp/xlnet-hinglish-chunk{chunk_idx + 1}")
    tokenizer.save_pretrained(f"temp/xlnet-hinglish-chunk{chunk_idx + 1}")

# Step 5: Final model save
model.save_pretrained("./xlnet-hinglish-final")
tokenizer.save_pretrained("./xlnet-hinglish-final")

In [ ]:
generator = pipeline("text-generation", model="./xlnet-hinglish-final", tokenizer=tokenizer)

user_input = input("Enter a prompt: ")
print(f"testing output for the input: {user_input}")
print(generator(user_input))